In [1]:
# ====== RAG Retrieval + Re-Ranker (Colab one-cell demo) ======
!pip -q install -U sentence-transformers chromadb

from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
import numpy as np

# -----------------------------
# 1) Sample documents (single or multiple files -> chunks)
# -----------------------------
docs = [
    ("doc1", "NTU add/drop deadline is typically in the early weeks of the semester. Check the official academic calendar."),
    ("doc2", "A prerequisite is a module you must complete before enrolling in an advanced module."),
    ("doc3", "Tuition fees may differ by programme and citizenship status. Refer to NTU fees page for exact values."),
    ("doc4", "For part-time students, plan weekly study blocks and include buffer time before quizzes and labs."),
    ("doc5", "Late course registration may require approval from the school office and may be subject to availability."),
]

# -----------------------------
# 2) Build vector store (Chroma) using a bi-encoder
# -----------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
col = client.get_or_create_collection("demo_rag")

ids = [d[0] for d in docs]
texts = [d[1] for d in docs]
embs = embed_model.encode(texts, normalize_embeddings=True).tolist()

# reset + add
try:
    col.delete(ids=ids)
except Exception:
    pass
col.add(ids=ids, documents=texts, embeddings=embs)

# -----------------------------
# 3) Retrieve top-k by embeddings
# -----------------------------
def retrieve_top_k(query: str, k: int = 4):
    q_emb = embed_model.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=k)
    # returns lists of lists
    return list(zip(res["ids"][0], res["documents"][0]))

# -----------------------------
# 4) Re-rank top-k with a cross-encoder (better precision)
# -----------------------------
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates, top_n: int = 2):
    pairs = [(query, text) for _id, text in candidates]
    scores = reranker.predict(pairs)  # higher = more relevant
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

# -----------------------------
# 5) Demo
# -----------------------------
query = "What is the add/drop deadline and what does prerequisite mean?"

top_k = retrieve_top_k(query, k=4)
top_n = rerank(query, top_k, top_n=3)

print("QUERY:", query)
print("\n--- Retrieved (top-k) ---")
for i, (doc_id, text) in enumerate(top_k, 1):
    print(f"{i}. {doc_id}: {text}")

print("\n--- Re-ranked (top-n) ---")
for i, (((doc_id, text), score)) in enumerate(top_n, 1):
    print(f"{i}. score={float(score):.4f} | {doc_id}: {text}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

QUERY: What is the add/drop deadline and what does prerequisite mean?

--- Retrieved (top-k) ---
1. doc1: NTU add/drop deadline is typically in the early weeks of the semester. Check the official academic calendar.
2. doc2: A prerequisite is a module you must complete before enrolling in an advanced module.
3. doc5: Late course registration may require approval from the school office and may be subject to availability.
4. doc4: For part-time students, plan weekly study blocks and include buffer time before quizzes and labs.

--- Re-ranked (top-n) ---
1. score=4.5536 | doc1: NTU add/drop deadline is typically in the early weeks of the semester. Check the official academic calendar.
2. score=2.3755 | doc2: A prerequisite is a module you must complete before enrolling in an advanced module.
3. score=-9.9251 | doc5: Late course registration may require approval from the school office and may be subject to availability.
